# 02 — PHM 2018 시간영역 특징 추출 및 정규화

이 노트북은 `00`의 환경 확인과 `01`의 데이터 구조 발견을 바탕으로 PHM 2018 Ion Mill 데이터의 **전처리와 정규화**를 수행합니다. JEV 또는 다른 모델은 학습하지 않습니다. 원본 `data/raw/`는 읽기만 하고 결과는 `data/interim/phm2018_jev/`에 파일별 CSV로 저장합니다.

## 수상작에서 가져온 전처리

PHM Society 2018 수상 논문은 웨이퍼 단위 시계열에서 13개 공정 변수를 선택하고, 100개 샘플 고정 창과 50% 이동 간격으로 평균, 표준편차, 피크, RMS, 첨도, 왜도, 크레스트 팩터, 형상 팩터를 계산했습니다. 이 노트북은 그 특징 추출 방식을 재현합니다. 논문의 Random Forest 건강 점수 및 DTW 열화 곡선 매칭 단계는 포함하지 않습니다. [수상 논문 원문](https://papers.phmsociety.org/index.php/phmconf/article/download/591/phmc_18_591)

## 처리 원칙

- 데이터에 있는 `runnum`을 연속 웨이퍼/공정 실행 식별자로 사용합니다. 창은 한 연속 `runnum` 구간 안에서만 만들고 서로 겹칩니다.
- 입력 후보는 센서 측정값 13개의 통계량(총 104개)입니다. `time`, `runnum`, `Lot`, `Tool`, TTF 열은 예측 특징으로 넣지 않습니다.
- 창 특징은 해당 창의 현재/과거 측정값만 사용합니다. train의 TTF는 창 마지막 시각 기준 정답 열로 보존하고, test에는 정답이 없으므로 비워 둡니다.
- 일부 원본 센서 값은 `01`의 샘플에서 이미 표준화된 범위로 보입니다. 원본 센서에 또 표준화를 적용하지 않고, 추출한 104개 특징만 장비별 z-score 정규화합니다.
- 장비별 운전 분포 차이를 고려해 정규화 통계는 `Tool`별 train 특징에서만 계산하고, 같은 `Tool`의 test에 재사용합니다. 정규화되지 않은 특징 표도 별도로 남깁니다.
- 정규화는 결측값을 채우지 않습니다. 이후 모델 검증 시 결측 대치가 필요하면 각 검증 fold의 train 부분에서만 대치값을 학습하세요.
- 창이 50%씩 겹치므로 이후 검증은 임의 행 분할 대신 `runnum` 단위 또는 시간 순서 분할을 사용해야 합니다.

정규화된 데이터는 JEV를 포함한 향후 분석에 사용할 수 있는 특징 산출물입니다. 모델/라벨 설계는 이 노트북의 범위에 포함하지 않습니다.


In [ ]:
from itertools import zip_longest
from pathlib import Path
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler
import json
import math
import numpy as np
import pandas as pd


## 1. 경로와 재현 가능한 설정

노트북을 프로젝트 루트나 `notebooks/`에서 열어도 프로젝트 루트를 찾습니다. `WINDOW_SIZE=100`은 데이터 샘플 수 기준입니다. `01`에서 확인된 약 4초 간격이 유지된다면 한 창은 약 400초이고, 50개 이동은 약 200초입니다. 시간 간격이 다른 파일에서도 창은 관측 샘플 수 기준으로 계산합니다.


In [ ]:
def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data" / "raw").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다. 노트북 커널의 작업 폴더를 확인하세요.")


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "phm_data_challenge_2018"
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
TTF_DIR = TRAIN_DIR / "train_ttf"
OUTPUT_ROOT = PROJECT_ROOT / "data" / "interim" / "phm2018_jev"
FEATURE_ROOT = OUTPUT_ROOT / "features"
NORMALIZED_ROOT = OUTPUT_ROOT / "normalized"

# 수상작과 같은 고정 창 특징 구성
WINDOW_SIZE = 100
WINDOW_STEP = 50
CHUNK_ROWS = 250_000
MIN_VALID_FRACTION = 0.80

SENSOR_FEATURES = [
    "IONGAUGEPRESSURE",
    "ETCHBEAMVOLTAGE",
    "ETCHBEAMCURRENT",
    "ETCHSUPPRESSORVOLTAGE",
    "ETCHSUPPRESSORCURRENT",
    "FLOWCOOLFLOWRATE",
    "FLOWCOOLPRESSURE",
    "ETCHGASCHANNEL1READBACK",
    "ETCHPBNGASREADBACK",
    "FIXTURETILTANGLE",
    "ROTATIONSPEED",
    "ACTUALROTATIONANGLE",
    "FIXTURESHUTTERPOSITION",
]

# TTF는 입력 특징이 아니라 train 행에 붙여 보존하는 정답 열입니다.
EXPECTED_TTF_COLUMNS = {
    "flowcool_pressure_low": "TTF_FlowCool Pressure Dropped Below Limit",
    "flowcool_pressure_high": "TTF_Flowcool Pressure Too High Check Flowcool Pump",
    "flowcool_leak": "TTF_Flowcool leak",
}

IDENTIFIER_COLUMNS = ["Tool", "stage", "Lot", "runnum", "recipe", "recipe_step"]
SENSOR_COLUMNS = ["time", *IDENTIFIER_COLUMNS, *SENSOR_FEATURES]
FEATURE_STATISTICS = [
    "mean", "std", "peak_abs", "rms", "kurtosis_excess",
    "skewness", "crest_factor", "shape_factor",
]
FEATURE_COLUMNS = [
    f"{sensor}__{stat}"
    for sensor in SENSOR_FEATURES
    for stat in FEATURE_STATISTICS
]

print("Project:", PROJECT_ROOT)
print("Raw data:", DATA_ROOT)
print("Output:", OUTPUT_ROOT)
print(f"Window: {WINDOW_SIZE} samples; step: {WINDOW_STEP} samples")


## 2. 스키마와 파일 짝 확인

각 train 센서 파일에는 같은 이름의 TTF 파일이 있어야 합니다. 센서와 TTF의 행별 `time`이 같은지 확인한 뒤, 같은 위치의 TTF를 결합합니다. 실제 데이터에는 동일 시각의 서로 다른 센서 관측이 있으므로 `time`만으로 병합하지 않습니다. test에는 정답이 없으므로 TTF를 비워 둡니다.


In [ ]:
train_sensor_files = sorted(TRAIN_DIR.glob("*.csv"))
test_sensor_files = sorted(TEST_DIR.glob("*.csv"))
if not train_sensor_files or not test_sensor_files:
    raise FileNotFoundError(
        f"센서 CSV를 찾을 수 없습니다. 확인할 경로: {TRAIN_DIR} 및 {TEST_DIR}"
    )

missing_ttf = [p.name for p in train_sensor_files if not (TTF_DIR / p.name).exists()]
if missing_ttf:
    raise FileNotFoundError(
        f"짝이 되는 TTF CSV가 없습니다: {missing_ttf[:5]}\n"
        "TTF 열을 추측하지 않고 실제 헤더와 파일 배치를 확인한 뒤 다시 실행하세요."
    )

sample_sensor_header = pd.read_csv(train_sensor_files[0], nrows=0).columns.tolist()
missing_sensor_columns = [c for c in SENSOR_COLUMNS if c not in sample_sensor_header]
if missing_sensor_columns:
    raise ValueError(f"예상한 센서 열이 없습니다: {missing_sensor_columns}")

def normalize_header(value: str) -> str:
    return " ".join(value.strip().casefold().split())

sample_ttf_header = pd.read_csv(TTF_DIR / train_sensor_files[0].name, nrows=0).columns.tolist()
ttf_header_lookup = {normalize_header(c): c for c in sample_ttf_header}
TTF_COLUMN_MAP = {}
for slug, expected_name in EXPECTED_TTF_COLUMNS.items():
    actual_name = ttf_header_lookup.get(normalize_header(expected_name))
    if actual_name is None:
        raise ValueError(
            f"예상한 TTF 열을 찾지 못했습니다: {expected_name}\n실제 열: {sample_ttf_header}"
        )
    TTF_COLUMN_MAP[slug] = actual_name

file_inventory = pd.DataFrame([
    {"split": "train", "sensor_files": len(train_sensor_files), "ttf_files": len(train_sensor_files)},
    {"split": "test", "sensor_files": len(test_sensor_files), "ttf_files": 0},
])
display(file_inventory)
print("입력 센서 열 수:", len(sample_sensor_header))
print("선택 센서 열 수:", len(SENSOR_FEATURES))
print("선택한 TTF 정답 열:", TTF_COLUMN_MAP)

# 입력값 자체의 대략적인 범위 확인용 예시(첫 train 파일의 앞 100,000행만 읽습니다).
raw_sample = pd.read_csv(
    train_sensor_files[0], usecols=SENSOR_FEATURES, nrows=100_000, low_memory=False
)
display(raw_sample.agg(["mean", "std", "min", "max"]).T.round(3))


## 3. 창 특징 계산

아래 통계량은 센서별 100개 관측값으로 계산합니다. 첨도는 정규분포가 0인 초과 첨도(Fisher), 피크는 절대값 최대, 크레스트 팩터는 `peak_abs / rms`, 형상 팩터는 `rms / mean_abs`로 정의했습니다. 센서 값이 창의 80% 미만만 유효하거나 분모가 0이면 해당 통계량은 결측으로 남깁니다.


In [ ]:
def calculate_window_features(sequence: pd.DataFrame):
    """연속 공정 실행 하나에서 겹치는 고정 길이 창과 통계 특징을 계산합니다."""
    if len(sequence) < WINDOW_SIZE:
        return np.empty((0,), dtype=int), np.empty((0, len(FEATURE_COLUMNS)), dtype=np.float32)

    values = sequence[SENSOR_FEATURES].to_numpy(dtype=np.float64, copy=False)
    # 결과 차원: (window, sensor, sample); 매 WINDOW_STEP 샘플마다 하나의 창을 택합니다.
    windows = sliding_window_view(values, WINDOW_SIZE, axis=0)[::WINDOW_STEP]
    finite = np.isfinite(windows)
    counts = finite.sum(axis=2).astype(np.float64)
    enough = counts >= math.ceil(WINDOW_SIZE * MIN_VALID_FRACTION)
    safe_counts = np.maximum(counts, 1.0)
    clean = np.where(finite, windows, 0.0)

    mean = clean.sum(axis=2) / safe_counts
    centered = np.where(finite, windows - mean[:, :, None], 0.0)
    moment2 = (centered ** 2).sum(axis=2) / safe_counts
    std = np.sqrt(moment2)
    moment3 = (centered ** 3).sum(axis=2) / safe_counts
    moment4 = (centered ** 4).sum(axis=2) / safe_counts
    peak_abs = np.max(np.where(finite, np.abs(windows), 0.0), axis=2)
    rms = np.sqrt((clean ** 2).sum(axis=2) / safe_counts)
    mean_abs = np.abs(clean).sum(axis=2) / safe_counts

    skewness = np.full_like(mean, np.nan)
    kurtosis_excess = np.full_like(mean, np.nan)
    crest_factor = np.full_like(mean, np.nan)
    shape_factor = np.full_like(mean, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        np.divide(moment3, std ** 3, out=skewness, where=std > 0)
        np.divide(moment4, std ** 4, out=kurtosis_excess, where=std > 0)
        kurtosis_excess -= 3.0
        np.divide(peak_abs, rms, out=crest_factor, where=rms > 0)
        np.divide(rms, mean_abs, out=shape_factor, where=mean_abs > 0)

    stats = np.stack(
        [mean, std, peak_abs, rms, kurtosis_excess, skewness, crest_factor, shape_factor],
        axis=2,
    )
    stats[~enough] = np.nan
    starts = np.arange(0, len(sequence) - WINDOW_SIZE + 1, WINDOW_STEP, dtype=int)
    return starts, stats.reshape(len(starts), -1).astype(np.float32)


def contiguous_run_slices(frame: pd.DataFrame):
    """runnum이 연속해서 같은 구간을 나눕니다. 같은 ID가 뒤에 재등장해도 합치지 않습니다."""
    run_key = frame["runnum"].astype("string").fillna("__MISSING_RUNNUM__")
    starts = np.flatnonzero(run_key.ne(run_key.shift()).fillna(True).to_numpy())
    ends = np.r_[starts[1:], len(frame)]
    return list(zip(starts.tolist(), ends.tolist()))


## 4. 파일별 청크 전처리

원본 센서 파일은 크므로 전체 파일을 한 번에 메모리에 올리지 않습니다. 창이 청크 경계나 웨이퍼 경계를 넘지 않도록 마지막 `runnum` 구간을 다음 청크까지 보류합니다. 동일 시각의 복수 센서 관측은 삭제하지 않고 원본 순서를 보존합니다. 센서와 TTF의 행별 시각이 일치하는지 검사한 뒤 같은 위치의 정답을 붙입니다. 이 단계에서는 아직 정규화하지 않은 특징 표를 만듭니다.


In [ ]:
OUTPUT_METADATA_COLUMNS = [
    "source_file", "Tool", "stage", "Lot", "runnum", "recipe", "recipe_step",
    "sequence_index", "window_index", "start_time", "end_time", "window_samples",
]
TTF_OUTPUT_COLUMNS = [f"ttf_{slug}_seconds" for slug in TTF_COLUMN_MAP]
OUTPUT_COLUMNS = [*OUTPUT_METADATA_COLUMNS, *FEATURE_COLUMNS, *TTF_OUTPUT_COLUMNS]


def process_sensor_file(sensor_path: Path, split: str) -> dict:
    """센서 파일 하나를 청크 단위로 처리하고 원시 특징 CSV를 씁니다."""
    is_train = split == "train"
    output_dir = FEATURE_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / f"{sensor_path.stem}_jev_windows.csv"
    ttf_path = (TTF_DIR / sensor_path.name) if is_train else None

    sensor_header = pd.read_csv(sensor_path, nrows=0).columns.tolist()
    absent = [c for c in SENSOR_COLUMNS if c not in sensor_header]
    if absent:
        raise ValueError(f"{sensor_path.name}: 센서 열이 없습니다: {absent}")

    sensor_reader = pd.read_csv(
        sensor_path, usecols=SENSOR_COLUMNS, chunksize=CHUNK_ROWS, low_memory=False
    )
    if is_train:
        ttf_reader = pd.read_csv(
            ttf_path,
            usecols=["time", *TTF_COLUMN_MAP.values()],
            chunksize=CHUNK_ROWS,
            low_memory=False,
        )
        chunk_pairs = zip_longest(sensor_reader, ttf_reader)
    else:
        chunk_pairs = ((sensor_chunk, None) for sensor_chunk in sensor_reader)

    carry = pd.DataFrame()
    previous_time = None
    output_started = False
    input_rows = 0
    row_count = 0
    window_count = 0
    sequence_count = 0
    short_sequence_count = 0
    duplicate_timestamp_rows = 0

    def emit_sequence(sequence: pd.DataFrame, records: list):
        nonlocal sequence_count, short_sequence_count, window_count
        this_sequence_index = sequence_count
        sequence_count += 1
        starts, feature_matrix = calculate_window_features(sequence)
        if len(starts) == 0:
            short_sequence_count += 1
            return

        seq_times = sequence["time"].to_numpy()
        for window_index, (sample_start, feature_row) in enumerate(zip(starts, feature_matrix)):
            end_position = int(sample_start + WINDOW_SIZE - 1)
            end_row = sequence.iloc[end_position]
            record = {
                "source_file": sensor_path.name,
                "Tool": end_row["Tool"],
                "stage": end_row["stage"],
                "Lot": end_row["Lot"],
                "runnum": end_row["runnum"],
                "recipe": end_row["recipe"],
                "recipe_step": end_row["recipe_step"],
                "sequence_index": this_sequence_index,
                "window_index": window_index,
                "start_time": seq_times[sample_start],
                "end_time": seq_times[end_position],
                "window_samples": WINDOW_SIZE,
            }
            record.update(zip(FEATURE_COLUMNS, feature_row.tolist()))
            for slug, ttf_column in TTF_COLUMN_MAP.items():
                record[f"ttf_{slug}_seconds"] = (
                    float(end_row[ttf_column])
                    if is_train and pd.notna(end_row[ttf_column])
                    else np.nan
                )
            records.append(record)
            window_count += 1

    for chunk_index, pair in enumerate(chunk_pairs, start=1):
        sensor_chunk, ttf_chunk = pair
        if sensor_chunk is None or (is_train and ttf_chunk is None):
            raise ValueError(f"{sensor_path.name}: 센서/TTF 청크 수가 서로 다릅니다.")

        times = sensor_chunk["time"].to_numpy()
        if len(times) == 0:
            continue
        time_deltas = np.diff(times)
        if np.any(time_deltas < 0):
            raise ValueError(f"{sensor_path.name} chunk {chunk_index}: time이 역행합니다.")
        duplicate_timestamp_rows += int(np.count_nonzero(time_deltas == 0))
        if previous_time is not None:
            if times[0] < previous_time:
                raise ValueError(f"{sensor_path.name} chunk {chunk_index}: 청크 사이 time이 역행합니다.")
            duplicate_timestamp_rows += int(times[0] == previous_time)
        previous_time = times[-1]
        input_rows += len(sensor_chunk)

        if is_train:
            if len(ttf_chunk) != len(sensor_chunk) or not np.array_equal(
                ttf_chunk["time"].to_numpy(), times
            ):
                raise ValueError(
                    f"{sensor_path.name} chunk {chunk_index}: 센서/TTF 행별 time이 일치하지 않습니다."
                )
            joined = pd.concat(
                [
                    sensor_chunk.reset_index(drop=True),
                    ttf_chunk.drop(columns="time").reset_index(drop=True),
                ],
                axis=1,
            )
        else:
            joined = sensor_chunk

        if not carry.empty:
            joined = pd.concat([carry, joined], ignore_index=True)
        sequences = contiguous_run_slices(joined)
        records = []
        # 마지막 연속 runnum은 다음 청크에서 이어질 수 있으므로 보류합니다.
        for start, end in sequences[:-1]:
            emit_sequence(joined.iloc[start:end], records)
        last_start, last_end = sequences[-1]
        carry = joined.iloc[last_start:last_end].copy()

        if records:
            batch = pd.DataFrame.from_records(records, columns=OUTPUT_COLUMNS)
            batch.to_csv(
                output_path,
                index=False,
                mode="a" if output_started else "w",
                header=not output_started,
            )
            output_started = True
            row_count += len(batch)

    if not carry.empty:
        records = []
        emit_sequence(carry, records)
        if records:
            batch = pd.DataFrame.from_records(records, columns=OUTPUT_COLUMNS)
            batch.to_csv(
                output_path,
                index=False,
                mode="a" if output_started else "w",
                header=not output_started,
            )
            output_started = True
            row_count += len(batch)

    if not output_started:
        pd.DataFrame(columns=OUTPUT_COLUMNS).to_csv(output_path, index=False)

    return {
        "split": split,
        "source_file": sensor_path.name,
        "input_rows": input_rows,
        "sequences": sequence_count,
        "short_sequences_skipped": short_sequence_count,
        "duplicate_timestamp_rows": duplicate_timestamp_rows,
        "windows_written": window_count,
        "output_rows": row_count,
        "output_file": str(output_path.relative_to(PROJECT_ROOT)),
    }


## 5. 특징 추출 실행

아래 셀은 train 20개 파일과 test 5개 파일을 차례로 처리합니다. 데이터 크기에 따라 실행 시간이 걸릴 수 있습니다. 같은 노트북을 다시 실행하면 파일별 원시 특징 표를 처음부터 다시 씁니다.


In [ ]:
summary_rows = []

for sensor_file in train_sensor_files:
    print("Train:", sensor_file.name)
    summary_rows.append(process_sensor_file(sensor_file, split="train"))

for sensor_file in test_sensor_files:
    print("Test :", sensor_file.name)
    summary_rows.append(process_sensor_file(sensor_file, split="test"))

preprocessing_summary = pd.DataFrame(summary_rows)
display(preprocessing_summary)


## 6. 장비별 train-only 정규화

평균과 표준편차는 정규화된 특징 파일을 대상으로, train split의 각 `Tool` 데이터에서만 추정합니다. 해당 `Tool`의 같은 통계를 train과 test 모두에 적용합니다. TTF와 식별 정보는 변환하지 않습니다. 결과는 원시 특징 표와 분리해 저장합니다.

이 전체-train 정규화 파일은 최종 학습/추론용입니다. 이후 성능을 검증할 때는 validation 기간의 통계가 scaler에 들어가지 않도록 fold별 train 부분에서 scaler를 다시 맞춰야 합니다.


In [ ]:
SCALER_CHUNK_ROWS = 100_000
raw_train_paths = sorted((FEATURE_ROOT / "train").glob("*_jev_windows.csv"))
raw_test_paths = sorted((FEATURE_ROOT / "test").glob("*_jev_windows.csv"))
if not raw_train_paths or not raw_test_paths:
    raise FileNotFoundError("원시 특징 표를 찾지 못했습니다. 먼저 5번 셀을 실행하세요.")

# 1) Tool별 유효값 수를 확인해 완전히 비어 있는 특징은 scaler에서 제외합니다.
nonnull_counts_by_tool = {}
for feature_path in raw_train_paths:
    for chunk in pd.read_csv(
        feature_path, usecols=["Tool", *FEATURE_COLUMNS],
        chunksize=SCALER_CHUNK_ROWS, low_memory=False,
    ):
        for raw_tool, group in chunk.groupby("Tool", sort=False, dropna=False):
            if pd.isna(raw_tool):
                raise ValueError(f"{feature_path.name}: Tool 식별값이 비어 있습니다.")
            tool = str(raw_tool)
            counts = group[FEATURE_COLUMNS].notna().sum()
            if tool not in nonnull_counts_by_tool:
                nonnull_counts_by_tool[tool] = counts.astype("int64")
            else:
                nonnull_counts_by_tool[tool] += counts.astype("int64")

active_features_by_tool = {
    tool: [c for c in FEATURE_COLUMNS if int(counts[c]) > 0]
    for tool, counts in nonnull_counts_by_tool.items()
}

# 2) StandardScaler.partial_fit으로 train 특징을 Tool별로 누적 학습합니다.
scalers_by_tool = {}
for feature_path in raw_train_paths:
    for chunk in pd.read_csv(
        feature_path, usecols=["Tool", *FEATURE_COLUMNS],
        chunksize=SCALER_CHUNK_ROWS, low_memory=False,
    ):
        for raw_tool, group in chunk.groupby("Tool", sort=False, dropna=False):
            tool = str(raw_tool)
            active = active_features_by_tool[tool]
            if not active or not group[active].notna().to_numpy().any():
                continue
            if tool not in scalers_by_tool:
                scalers_by_tool[tool] = StandardScaler()
            scalers_by_tool[tool].partial_fit(group[active])

missing_scalers = sorted(set(active_features_by_tool) - set(scalers_by_tool))
if missing_scalers:
    raise ValueError(f"유효 특징이 있으나 scaler가 만들어지지 않은 Tool: {missing_scalers}")

# 3) Tool별 train scaler를 train/test 원시 특징에 동일하게 적용합니다.
normalization_rows = []
for split, input_paths in [("train", raw_train_paths), ("test", raw_test_paths)]:
    output_dir = NORMALIZED_ROOT / split
    output_dir.mkdir(parents=True, exist_ok=True)
    for feature_path in input_paths:
        output_path = output_dir / feature_path.name
        wrote = False
        normalized_rows = 0
        for chunk in pd.read_csv(feature_path, chunksize=SCALER_CHUNK_ROWS, low_memory=False):
            for raw_tool, row_indices in chunk.groupby("Tool", sort=False, dropna=False).groups.items():
                if pd.isna(raw_tool):
                    raise ValueError(f"{feature_path.name}: Tool 식별값이 비어 있습니다.")
                tool = str(raw_tool)
                if tool not in scalers_by_tool:
                    raise ValueError(
                        f"{feature_path.name}: {tool}의 train scaler가 없습니다."
                        " 이 변환은 train에서 확인된 Tool에만 적용할 수 있습니다."
                    )
                active = active_features_by_tool[tool]
                if active:
                    chunk.loc[row_indices, active] = scalers_by_tool[tool].transform(
                        chunk.loc[row_indices, active]
                    )
            chunk.to_csv(
                output_path, index=False,
                mode="a" if wrote else "w", header=not wrote,
            )
            wrote = True
            normalized_rows += len(chunk)
        normalization_rows.append({
            "split": split,
            "source_file": feature_path.name,
            "normalized_rows": normalized_rows,
            "normalized_file": str(output_path.relative_to(PROJECT_ROOT)),
        })

def json_number(value):
    return float(value) if np.isfinite(value) else None

scaler_metadata = {
    "method": "per-Tool StandardScaler; z = (x - train_mean) / train_scale",
    "fit_split": "train only",
    "feature_columns": FEATURE_COLUMNS,
    "window_size_samples": WINDOW_SIZE,
    "window_step_samples": WINDOW_STEP,
    "tools": {},
}
for tool, scaler in scalers_by_tool.items():
    active = active_features_by_tool[tool]
    scaler_metadata["tools"][tool] = {
        "n_train_windows": int(nonnull_counts_by_tool[tool].max()),
        "features": {
            feature: {
                "mean": json_number(scaler.mean_[i]),
                "scale": json_number(scaler.scale_[i]),
            }
            for i, feature in enumerate(active)
        },
        "all_missing_features": [c for c in FEATURE_COLUMNS if c not in active],
    }

scaler_path = OUTPUT_ROOT / "normalization_parameters_by_tool.json"
scaler_path.parent.mkdir(parents=True, exist_ok=True)
scaler_path.write_text(
    json.dumps(scaler_metadata, ensure_ascii=False, indent=2), encoding="utf-8"
)
normalization_summary = pd.DataFrame(normalization_rows)
display(normalization_summary)
print("Tool별 정규화 파라미터:", scaler_path.relative_to(PROJECT_ROOT))


## 7. 산출물

- 원시 시간영역 특징: `data/interim/phm2018_jev/features/train/` 및 `test/`
- 장비별 정규화 특징: `data/interim/phm2018_jev/normalized/train/` 및 `test/`
- Tool별 train 평균/척도: `data/interim/phm2018_jev/normalization_parameters_by_tool.json`

정규화 표에서 104개 특징만 모델 입력 후보로 사용합니다. `ttf_*_seconds`는 train에서만 존재하는 정답이며 입력 특징이 아닙니다. 결측값은 그대로 유지됩니다. 이후 모델 검증에서는 전체 train으로 미리 계산된 정규화 표를 validation 평가에 그대로 쓰지 말고, 각 fold의 train 부분에서 정규화 통계를 다시 맞추세요.


In [ ]:
display(preprocessing_summary)
display(normalization_summary)

preview_path = sorted((NORMALIZED_ROOT / "train").glob("*_jev_windows.csv"))[0]
preview_columns = [
    "Tool", "runnum", "start_time", "end_time",
    "FLOWCOOLFLOWRATE__mean", "FLOWCOOLPRESSURE__mean",
    *TTF_OUTPUT_COLUMNS,
]
preview = pd.read_csv(preview_path, usecols=preview_columns, nrows=5)
display(preview)
print("정규화된 특징 열 수:", len(FEATURE_COLUMNS))
print("출력 루트:", OUTPUT_ROOT)
